# LangChain RAG with ChromaDB — Loaders, Chunkers, and LCEL

A complete walkthrough of building a Retrieval-Augmented Generation pipeline with the latest LangChain (v0.3+) APIs.

**Stack**
- `langchain` v0.3+ (current modular packages: `langchain-openai`, `langchain-chroma`, `langchain-community`, `langchain-text-splitters`)
- **ChromaDB** for vector storage
- **text-embedding-3-small** for embeddings
- **gpt-4o-mini** for generation
- **LCEL** (LangChain Expression Language) for declarative pipeline composition

**What this notebook covers**
1. Multiple document loaders — PDF, web, text, CSV, directory
2. Three chunking strategies side by side — recursive, character, token-based
3. Embedding and indexing in Chroma with metadata
4. Different retriever types — similarity, MMR, similarity-with-threshold
5. The classic RAG chain built with LCEL primitives
6. A history-aware RAG chain that handles follow-up questions
7. Streaming the answer token-by-token

Every section prints what it produced so you can see what is going on at every stage.

## 1. Install Dependencies

In [8]:
!pip install -q \
    langchain==0.3.27 \
    langchain-core==0.3.78 \
    langchain-openai==0.3.34 \
    langchain-chroma==0.2.6 \
    langchain-community==0.3.31 \
    langchain-text-splitters==0.3.11 \
    chromadb==1.3.0 \
    pypdf==6.1.3 \
    tiktoken==0.12.0 \
    beautifulsoup4==4.14.2 \
    lxml==6.0.2

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 25.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 449.6/449.6 kB 30.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.6/75.6 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 70.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.8/20.8 MB 51.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.9/323.9 kB 22.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 106.4/106.4 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 101.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 72.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 83.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.

## 2. Load Secrets from Colab

Add `OPENAI_API_KEY` in the Colab secrets panel (left sidebar → key icon → toggle "Notebook access").

In [9]:
import os
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

# Disable LangSmith tracing by default — flip to "true" if you want to trace.
os.environ["LANGCHAIN_TRACING_V2"] = "false"

print("OpenAI key loaded:", bool(os.environ.get("OPENAI_API_KEY")))

OpenAI key loaded: True


## 3. Initialize the LLM and the Embedding Model

`langchain-openai` provides thin wrappers around the official OpenAI client. The same `ChatOpenAI` and `OpenAIEmbeddings` objects are used everywhere — passed into chains, retrievers, etc.

In [13]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
import os
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.1
)

embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small"
)

vec = embeddings.embed_query("test")

print("Embedding dimensions:", len(vec))

Embedding dimensions: 1536


## 4. Document Loaders

LangChain has loaders for almost every source. Each returns a list of `Document` objects, where a `Document` has `page_content` (text) and `metadata` (dict).

We will demonstrate five of the most common loaders.

### 4a. PDF Loader (`PyPDFLoader`)

Loads a PDF and creates one `Document` per page. Page numbers land in metadata automatically.

In [14]:
from google.colab import files

print("Upload a PDF file:")
uploaded = files.upload()
pdf_path = list(uploaded.keys())[0]
print(f"\nUploaded: {pdf_path}")

Upload a PDF file:


Saving The_Brain_behind_AI (1).pdf to The_Brain_behind_AI (1).pdf

Uploaded: The_Brain_behind_AI (1).pdf


In [15]:
from langchain_community.document_loaders import PyPDFLoader

pdf_loader = PyPDFLoader(pdf_path)
pdf_docs = pdf_loader.load()

print(f"Loaded {len(pdf_docs)} page-documents from the PDF\n")
print("First document preview:")
print(f"  page_content[:200] = {pdf_docs[0].page_content[:200]!r}")
print(f"  metadata          = {pdf_docs[0].metadata}")

Loaded 15 page-documents from the PDF

First document preview:
  page_content[:200] = 'The Brain behind AI\nAuthor: AI Generated eBook'
  metadata          = {'producer': 'ReportLab PDF Library - www.reportlab.com', 'creator': '(unspecified)', 'creationdate': '2025-10-27T19:17:04-05:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2025-10-27T19:17:04-05:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': 'The_Brain_behind_AI (1).pdf', 'total_pages': 15, 'page': 0, 'page_label': '1'}


### 4b. Web Page Loader (`WebBaseLoader`)

Fetches an HTML page and extracts the visible text. Useful for ingesting documentation sites, blog posts, or any public URL.

In [16]:
from langchain_community.document_loaders import WebBaseLoader

# Example — replace with your own URL.
web_loader = WebBaseLoader(["https://python.langchain.com/docs/introduction/"])
web_docs = web_loader.load()

print(f"Loaded {len(web_docs)} web document(s)\n")
print("First document preview:")
print(f"  page_content[:300] = {web_docs[0].page_content[:300]!r}")
print(f"  metadata          = {web_docs[0].metadata}")

Loaded 1 web document(s)

First document preview:
  page_content[:300] = 'LangChain overview - Docs by LangChainSkip to main contentJoin us May 13th & May 14th at Interrupt, the Agent Conference by LangChain. Buy tickets >Docs by LangChain home pageOpen sourceSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationLangChain overviewDeep AgentsLangChainLangGrap'
  metadata          = {'source': 'https://python.langchain.com/docs/introduction/', 'title': 'LangChain overview - Docs by LangChain', 'description': 'LangChain is an open source framework with a prebuilt agent architecture and integrations for any model or tool—so you can build agents that adapt as fast as the ecosystem evolves', 'language': 'en'}


### 4c. Plain Text Loader (`TextLoader`)

The simplest loader — reads a `.txt` file as a single `Document`.

In [17]:
# Create a sample text file to demonstrate.
sample_text = '''ChromaDB is an open-source embedding database designed for RAG applications.
It supports multiple embedding models and provides fast similarity search.
Chroma can run in-memory or be persisted to disk for production use.

LangChain integrates with Chroma through the langchain-chroma package,
which provides a Vectorstore interface for adding, querying, and managing documents.'''

with open("/tmp/sample.txt", "w") as f:
    f.write(sample_text)

from langchain_community.document_loaders import TextLoader
text_loader = TextLoader("/tmp/sample.txt")
text_docs = text_loader.load()

print(f"Loaded {len(text_docs)} text document(s)\n")
print(f"  page_content = {text_docs[0].page_content!r}")
print(f"  metadata     = {text_docs[0].metadata}")

Loaded 1 text document(s)

  page_content = 'ChromaDB is an open-source embedding database designed for RAG applications.\nIt supports multiple embedding models and provides fast similarity search.\nChroma can run in-memory or be persisted to disk for production use.\n\nLangChain integrates with Chroma through the langchain-chroma package,\nwhich provides a Vectorstore interface for adding, querying, and managing documents.'
  metadata     = {'source': '/tmp/sample.txt'}


### 4d. CSV Loader (`CSVLoader`)

Treats each row of a CSV as a separate `Document`. Each column becomes a `key: value` line in the row's text. Great for FAQ tables, product catalogs, or any tabular knowledge.

In [18]:
# Create a sample CSV.
csv_content = '''product_id,name,description,price
P-001,Cloud Storage Pro,Secure cloud storage with 2TB capacity and team sharing,49.99
P-002,Email Suite,Business email with custom domains and 100GB inbox,12.99
P-003,Analytics Plus,Real-time analytics dashboard with custom reports,79.99'''

with open("/tmp/products.csv", "w") as f:
    f.write(csv_content)

from langchain_community.document_loaders import CSVLoader
csv_loader = CSVLoader("/tmp/products.csv")
csv_docs = csv_loader.load()

print(f"Loaded {len(csv_docs)} row-documents from the CSV\n")
for d in csv_docs:
    print(f"  page_content = {d.page_content!r}")
    print(f"  metadata     = {d.metadata}")
    print()

Loaded 3 row-documents from the CSV

  page_content = 'product_id: P-001\nname: Cloud Storage Pro\ndescription: Secure cloud storage with 2TB capacity and team sharing\nprice: 49.99'
  metadata     = {'source': '/tmp/products.csv', 'row': 0}

  page_content = 'product_id: P-002\nname: Email Suite\ndescription: Business email with custom domains and 100GB inbox\nprice: 12.99'
  metadata     = {'source': '/tmp/products.csv', 'row': 1}

  page_content = 'product_id: P-003\nname: Analytics Plus\ndescription: Real-time analytics dashboard with custom reports\nprice: 79.99'
  metadata     = {'source': '/tmp/products.csv', 'row': 2}



### 4e. Directory Loader (`DirectoryLoader`)

Recursively loads every matching file in a folder. Useful when your knowledge base is a folder of mixed files.

In [19]:
# Set up a small directory with two text files to demonstrate.
import os
os.makedirs("/tmp/kb", exist_ok=True)
with open("/tmp/kb/policy.txt", "w") as f:
    f.write("Refunds are processed within 5 business days.")
with open("/tmp/kb/shipping.txt", "w") as f:
    f.write("Standard shipping takes 3-5 days. Express shipping takes 1-2 days.")

from langchain_community.document_loaders import DirectoryLoader, TextLoader
dir_loader = DirectoryLoader(
    "/tmp/kb",
    glob="**/*.txt",
    loader_cls=TextLoader,
    show_progress=True,
)
dir_docs = dir_loader.load()

print(f"\nLoaded {len(dir_docs)} documents from the directory:\n")
for d in dir_docs:
    print(f"  source: {d.metadata['source']}")
    print(f"  text  : {d.page_content!r}")
    print()

100%|██████████| 2/2 [00:00<00:00, 2082.06it/s]


Loaded 2 documents from the directory:

  source: /tmp/kb/shipping.txt
  text  : 'Standard shipping takes 3-5 days. Express shipping takes 1-2 days.'

  source: /tmp/kb/policy.txt
  text  : 'Refunds are processed within 5 business days.'



## 5. Chunking Strategies

Splitters break long documents into smaller chunks that fit in an embedding context and produce focused retrieval results. LangChain offers several strategies — choosing the right one matters more than people expect.

We will compare three on the PDF you uploaded:

1. **CharacterTextSplitter** — splits on a single separator (simple, sometimes too coarse)
2. **RecursiveCharacterTextSplitter** — tries separators in order, preserves structure best (the default recommendation)
3. **TokenTextSplitter** — splits by tiktoken tokens, exact for LLM context limits

### 5a. CharacterTextSplitter

Splits on a single separator (default: `\n\n`). Fast but can leave chunks much larger than `chunk_size` if the separator is rare.

In [20]:
from langchain_text_splitters import CharacterTextSplitter

char_splitter = CharacterTextSplitter(
    separator="\n\n",
    chunk_size=500,
    chunk_overlap=50,
    length_function=len,
)
char_chunks = char_splitter.split_documents(pdf_docs)

print(f"[CharacterTextSplitter] {len(char_chunks)} chunks")
print(f"  avg length: {sum(len(c.page_content) for c in char_chunks) // max(1, len(char_chunks))} chars")
print(f"\nFirst chunk:\n{char_chunks[0].page_content[:300]}")

[CharacterTextSplitter] 15 chunks
  avg length: 3426 chars

First chunk:
The Brain behind AI
Author: AI Generated eBook


### 5b. RecursiveCharacterTextSplitter (Recommended Default)

Tries separators in order (`\n\n`, `\n`, `. `, ` `, `""`). Falls back gracefully when one separator is missing. Produces more uniform chunk sizes than `CharacterTextSplitter`.

In [21]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    separators=["\n\n", "\n", ". ", " ", ""],
    length_function=len,
)
recursive_chunks = recursive_splitter.split_documents(pdf_docs)

print(f"[RecursiveCharacterTextSplitter] {len(recursive_chunks)} chunks")
print(f"  avg length: {sum(len(c.page_content) for c in recursive_chunks) // max(1, len(recursive_chunks))} chars")
print(f"\nFirst chunk:\n{recursive_chunks[0].page_content[:300]}")

[RecursiveCharacterTextSplitter] 119 chunks
  avg length: 430 chars

First chunk:
The Brain behind AI
Author: AI Generated eBook


### 5c. TokenTextSplitter

Counts in **tokens** using `tiktoken`. Use this when chunk size matters in token terms (e.g. fitting into a model's context budget). For OpenAI embedding models, 1 token ≈ 4 characters of English.

In [22]:
from langchain_text_splitters import TokenTextSplitter

token_splitter = TokenTextSplitter(
    chunk_size=200,        # 200 tokens, not characters
    chunk_overlap=20,
    encoding_name="cl100k_base",  # used by text-embedding-3-* and gpt-4o-*
)
token_chunks = token_splitter.split_documents(pdf_docs)

print(f"[TokenTextSplitter] {len(token_chunks)} chunks")
print(f"  avg length (chars): {sum(len(c.page_content) for c in token_chunks) // max(1, len(token_chunks))}")
print(f"\nFirst chunk:\n{token_chunks[0].page_content[:300]}")

[TokenTextSplitter] 73 chunks
  avg length (chars): 769

First chunk:
The Brain behind AI
Author: AI Generated eBook


### 5d. Compare the Three Strategies

In [23]:
import statistics

def chunk_stats(chunks, name):
    lengths = [len(c.page_content) for c in chunks]
    return {
        "Strategy": name,
        "Total chunks": len(chunks),
        "Min chars": min(lengths) if lengths else 0,
        "Max chars": max(lengths) if lengths else 0,
        "Mean chars": int(statistics.mean(lengths)) if lengths else 0,
        "Std dev": int(statistics.stdev(lengths)) if len(lengths) > 1 else 0,
    }

import pandas as pd
df = pd.DataFrame([
    chunk_stats(char_chunks, "Character"),
    chunk_stats(recursive_chunks, "Recursive"),
    chunk_stats(token_chunks, "Token"),
])
df

,Strategy,Total chunks,Min chars,Max chars,Mean chars,Std dev
0,Character,15,46,4766,3426,1668
1,Recursive,119,44,499,430,77
2,Token,73,46,972,769,179


**Takeaways**

| Strategy | Best for |
|---|---|
| **Character** | Documents with a strong, consistent separator (e.g. markdown sections separated by blank lines) |
| **Recursive** | General-purpose default. Handles varied document structure gracefully |
| **Token** | When you need precise control over LLM context budget |

For the rest of this notebook we use the **recursive** chunks because they are the most uniform.

## 6. Build the Chroma Vector Index

`langchain-chroma` exposes Chroma through the standard `VectorStore` interface. `from_documents` embeds all chunks and adds them in one call. Pass `persist_directory` to keep the index on disk between runs.

In [24]:
from langchain_chroma import Chroma

# Wipe any prior index for a clean run.
import shutil
shutil.rmtree("/tmp/chroma_db", ignore_errors=True)

print(f"Embedding and indexing {len(recursive_chunks)} chunks in Chroma...")
vectorstore = Chroma.from_documents(
    documents=recursive_chunks,
    embedding=embeddings,
    persist_directory="/tmp/chroma_db",
    collection_name="rag_demo",
)

# Sanity check.
count = vectorstore._collection.count()
print(f"Chroma collection now contains {count} vectors")

Embedding and indexing 119 chunks in Chroma...
Chroma collection now contains 119 vectors


## 7. Retriever Variations

A retriever is just `vectorstore.as_retriever(...)` with a search strategy.

| `search_type` | Behavior |
|---|---|
| `"similarity"` | Pure cosine similarity (default). Fastest. |
| `"mmr"` | Maximal Marginal Relevance. Balances relevance with diversity — useful when top results are too redundant. |
| `"similarity_score_threshold"` | Only returns docs above a score threshold. Useful when "no result" should be an option. |

In [25]:
TEST_QUERY = "What is this document about?"

# Plain similarity
sim_retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4},
)

# Maximal Marginal Relevance — fetches a wider pool then diversifies.
mmr_retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 4, "fetch_k": 20, "lambda_mult": 0.5},
)

# Threshold — only returns results above 0.3 cosine similarity.
threshold_retriever = vectorstore.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={"k": 4, "score_threshold": 0.3},
)

def show_retrieval(name, retriever, query):
    print("=" * 80)
    print(f"{name}  |  query: {query!r}")
    print("=" * 80)
    docs = retriever.invoke(query)
    for i, d in enumerate(docs, 1):
        page = d.metadata.get("page", "?")
        preview = d.page_content[:150].replace("\n", " ")
        print(f"  [{i}] page={page}  {preview}...")
    if not docs:
        print("  (no results above threshold)")
    print()

show_retrieval("Similarity",          sim_retriever,       TEST_QUERY)
show_retrieval("MMR (diversity)",     mmr_retriever,       TEST_QUERY)
show_retrieval("Similarity threshold", threshold_retriever, TEST_QUERY)

Similarity  |  query: 'What is this document about?'
  [1] page=3  2017 “Attention Is All You Need” paper, a quiet earthquake in the AI world. > *Example: How Google Translate’s shift from phrase-based to neural model...
  [2] page=11  Common Crawl (a massive web archive), Reddit, Wikipedia, and books. Much of this data was created by humans—for humans—not for machines to ingest and ...
  [3] page=1  technical manual, nor a dystopian prophecy. It is a journey—a guided exploration into the convergence of neuroscience, computer science, psychology, a...
  [4] page=9  Chapter 3 # **Chapter 3: From Language to Vision — The Transformer’s Empire Expands** ### *When Words Became Images, Code, and Molecules* > *“The same...

MMR (diversity)  |  query: 'What is this document about?'
  [1] page=3  2017 “Attention Is All You Need” paper, a quiet earthquake in the AI world. > *Example: How Google Translate’s shift from phrase-based to neural model...
  [2] page=11  Common Crawl (a massive web arch

/usr/local/lib/python3.12/dist-packages/langchain_core/vectorstores/base.py:1070: UserWarning: Relevance scores must be between 0 and 1, got [(Document(id='9fe620fc-fefa-4d2f-aa01-521379910d58', metadata={'page_label': '4', 'total_pages': 15, 'source': 'The_Brain_behind_AI (1).pdf', 'creator': '(unspecified)', 'creationdate': '2025-10-27T19:17:04-05:00', 'trapped': '/False', 'page': 3, 'subject': '(unspecified)', 'author': '(anonymous)', 'keywords': '', 'title': '(anonymous)', 'producer': 'ReportLab PDF Library - www.reportlab.com', 'moddate': '2025-10-27T19:17:04-05:00'}, page_content='2017 “Attention Is All You Need” paper, a quiet earthquake in the AI world. > *Example: How Google\nTranslate’s shift from phrase-based to neural models foreshadowed the transformer era.* --- ###\n**Chapter 2: The Anatomy of Attention — How Transformers See the World** *Decoding the Core\nMechanism That Changed Everything* Here, we dissect the transformer’s revolutionary engine:\n**self-attention**. Thr

  (no results above threshold)



## 8. LCEL — LangChain Expression Language

LCEL composes chains with the `|` operator. Every piece is a `Runnable` that has a consistent `.invoke`, `.batch`, `.stream`, and `.ainvoke` interface.

### 8a. The simplest LCEL chain

A prompt template piped into the LLM piped into an output parser. No retrieval yet — just to see the LCEL shape.

In [26]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a concise assistant. Answer in one sentence."),
    ("human", "{question}"),
])

simple_chain = prompt | llm | StrOutputParser()

result = simple_chain.invoke({"question": "What is RAG?"})
print(result)

RAG stands for Retrieval-Augmented Generation, a technique that combines information retrieval with generative models to enhance the quality and relevance of generated text.


## 9. RAG Chain Built with LCEL

The classic RAG pipeline expressed in LCEL primitives:

```
{question}
   |
   ├── retriever  → context (list of Documents)
   └── itself     → question (str)
   |
prompt template  ←  fills in {context} and {question}
   |
   LLM
   |
parser  →  final answer string
```

`RunnableParallel` lets us run two branches in parallel — one fetches context, the other passes the question through unchanged.

In [27]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel, RunnablePassthrough

# 1. The prompt — note the {context} and {question} placeholders.
rag_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a careful assistant. Answer the user's question strictly from the "
     "provided context. If the answer is not in the context, say you don't know. "
     "Cite the page numbers you used in square brackets like [page 3]."),
    ("human",
     "Context:\n{context}\n\nQuestion: {question}")
])

# 2. Helper to format retrieved documents into a single context string.
def format_docs(docs):
    return "\n\n---\n\n".join(
        f"[page {d.metadata.get('page', '?')}] {d.page_content}"
        for d in docs
    )

# 3. Compose the chain.
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

rag_chain = (
    RunnableParallel(
        context=retriever | format_docs,
        question=RunnablePassthrough(),
    )
    | rag_prompt
    | llm
    | StrOutputParser()
)

# 4. Invoke it.
question = "What is the main topic of this document?"
answer = rag_chain.invoke(question)
print("Question:", question)
print("\nAnswer:")
print(answer)

Question: What is the main topic of this document?

Answer:
The main topic of the document is the transformative impact of the "Attention Is All You Need" paper on artificial intelligence, particularly focusing on the mechanisms of transformers, their applications across various domains, and the associated societal implications and ethical considerations [page 3].


### 9a. Inspect the Retrieved Context

A subtle but important debugging tip: the same `RunnableParallel` can be invoked on its own to see exactly what context the LLM was given.

In [28]:
inspect_chain = RunnableParallel(
    context=retriever | format_docs,
    question=RunnablePassthrough(),
)
result = inspect_chain.invoke(question)
print("Retrieved context the LLM saw:\n")
print(result["context"])

Retrieved context the LLM saw:

[page 3] 2017 “Attention Is All You Need” paper, a quiet earthquake in the AI world. > *Example: How Google
Translate’s shift from phrase-based to neural models foreshadowed the transformer era.* --- ###
**Chapter 2: The Anatomy of Attention — How Transformers See the World** *Decoding the Core
Mechanism That Changed Everything* Here, we dissect the transformer’s revolutionary engine:
**self-attention**. Through intuitive visuals and step-by-step explanations, we reveal how attention

---

[page 3] data hunger that fuels surveillance concerns, and the amplification of societal biases (e.g., racial or
gender stereotypes in generated text). We profile emerging solutions: sparse attention, distillation, and
ethical AI frameworks. The chapter balances optimism with realism, asking: *Can we build intelligence
without burning the planet or eroding trust?* > *Case Study: The controversy around Tay, Microsoft’s

---

[page 9] Chapter 3
# **Chapter 3: From Langua

## 10. RAG Chain that Returns Sources

For production use we usually want the answer **and** the documents that produced it. `RunnableParallel` makes this trivial — fan out to two paths and return both.

In [29]:
from operator import itemgetter

rag_with_sources = RunnableParallel(
    context=retriever,
    question=RunnablePassthrough(),
).assign(
    answer=(
        {
            "context": lambda x: format_docs(x["context"]),
            "question": itemgetter("question"),
        }
        | rag_prompt
        | llm
        | StrOutputParser()
    )
)

out = rag_with_sources.invoke("What is the main topic of this document?")

print("Answer:\n")
print(out["answer"])
print("\n" + "=" * 80)
print("Sources used:\n")
for i, d in enumerate(out["context"], 1):
    print(f"  [{i}] page={d.metadata.get('page')} — {d.page_content[:120].strip()}...")

Answer:

The main topic of the document is the transformative impact of the transformer architecture in artificial intelligence, particularly focusing on mechanisms like self-attention and its applications across various domains, including language processing and vision. It discusses the evolution of AI models, ethical considerations, and the broader implications of these advancements [page 3, page 7, page 9].

Sources used:

  [1] page=3 — 2017 “Attention Is All You Need” paper, a quiet earthquake in the AI world. > *Example: How Google
Translate’s shift fro...
  [2] page=3 — data hunger that fuels surveillance concerns, and the amplification of societal biases (e.g., racial or
gender stereotyp...
  [3] page=9 — Chapter 3
# **Chapter 3: From Language to Vision — The Transformer’s Empire Expands** ### *When Words
Became Images, Cod...
  [4] page=7 — Chapter 2
# **Chapter 2: The Anatomy of Attention — How Transformers See the World** ### *Decoding the
Core Mechanism Th...


## 11. History-Aware RAG (Follow-Up Questions)

When the user asks a follow-up like "what about chapter 3?", the retrieval step needs the conversation history to rephrase the query into something self-contained.

LangChain ships `create_history_aware_retriever` for exactly this: it uses the LLM to rewrite the follow-up into a standalone question, then runs the retriever on that.

In [30]:
from langchain.chains import create_history_aware_retriever, create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

# Prompt that rewrites a follow-up into a standalone query.
condense_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "Given the chat history and the latest user question, rewrite the question "
     "so it is fully self-contained. Do NOT answer it — only rewrite. "
     "If it is already standalone, return it as-is."),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])

history_aware_retriever = create_history_aware_retriever(
    llm=llm,
    retriever=retriever,
    prompt=condense_prompt,
)

# Prompt that answers using retrieved context.
qa_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "Answer the user's question strictly from the context below. "
     "If the answer is not present, say you don't know. Cite page numbers.\n\n"
     "Context:\n{context}"),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])

qa_chain = create_stuff_documents_chain(llm, qa_prompt)
conversational_rag = create_retrieval_chain(history_aware_retriever, qa_chain)

# Demo: a turn, then a follow-up that needs the history to be understood.
from langchain_core.messages import HumanMessage, AIMessage

chat_history = []

# Turn 1
q1 = "What is the main topic of this document?"
r1 = conversational_rag.invoke({"input": q1, "chat_history": chat_history})
print(f"USER: {q1}")
print(f"BOT : {r1['answer']}\n")
chat_history.extend([HumanMessage(content=q1), AIMessage(content=r1["answer"])])

# Turn 2 — vague follow-up
q2 = "Can you elaborate on that?"
r2 = conversational_rag.invoke({"input": q2, "chat_history": chat_history})
print(f"USER: {q2}")
print(f"BOT : {r2['answer']}")

USER: What is the main topic of this document?
BOT : The main topic of the document is the transformative impact of the "Attention Is All You Need" paper on artificial intelligence, particularly focusing on the development and implications of the transformer architecture in various applications, including language processing and beyond. It discusses the mechanisms of self-attention, the expansion of transformers into different domains, and the associated societal concerns and ethical considerations (Chapter 2 and Chapter 3).

USER: Can you elaborate on that?
BOT : The document elaborates on the revolutionary impact of the transformer architecture introduced in the 2017 paper "Attention Is All You Need." It highlights how this architecture, built entirely on self-attention mechanisms, marked a significant departure from previous AI models that relied on recurrent or convolutional structures. This shift has led to a major transformation in natural language processing (NLP) and has expand

## 12. Streaming the Answer

Every LCEL chain supports `.stream()` for free. The chain we built in section 9 streams tokens as the LLM produces them.

In [32]:
print("Streaming answer:\n")
for chunk in rag_chain.stream("What is this document about?"):
    print(chunk, end="", flush=True)
print("\n\n(stream complete)")

Streaming answer:

This document appears to be about the transformative impact of the transformer architecture in artificial intelligence, particularly focusing on the concept of self-attention and its applications across various fields such as language processing, vision, and more. It discusses the convergence of different disciplines like neuroscience, computer science, psychology, and philosophy, and raises ethical questions regarding data usage in AI models [page 1][page 3][page 11].

(stream complete)


## 13. Batch Mode

Need to answer many questions at once? `.batch()` runs them concurrently — much faster than a Python loop.

In [31]:
questions = [
    "What is the main topic of the document?",
    "Who is the intended audience?",
    "Are there any key terms or definitions?",
]

answers = rag_chain.batch(questions)

for q, a in zip(questions, answers):
    print(f"Q: {q}")
    print(f"A: {a}\n")

Q: What is the main topic of the document?
A: The main topic of the document revolves around the transformative impact of the "Attention Is All You Need" paper and the subsequent development of transformer models in AI, particularly focusing on their applications, ethical considerations, and the challenges related to data usage and societal biases [page 3][page 9][page 11].

Q: Who is the intended audience?
A: The intended audience includes tech-savvy readers, professionals, and curious minds—no PhD required [page 4].

Q: Are there any key terms or definitions?
A: Yes, there are key terms and definitions provided in the context. Some of them include:

1. **Self-attention**: A mathematical process used by Transformers to focus on different parts of the input data.
2. **Query (Q)**: Represents "What am I looking for?"
3. **Key (K)**: Represents "What do I offer?"
4. **Value (V)**: Represents "What information do I carry?"
5. **Attention heads**: Parallel components in the model that lear

## 14. Summary

| Layer | What we used | Where to extend |
|---|---|---|
| **Loaders** | PyPDFLoader, WebBaseLoader, TextLoader, CSVLoader, DirectoryLoader | `langchain_community.document_loaders` has 100+ more (Notion, Slack, S3, GitHub, etc.) |
| **Splitters** | CharacterTextSplitter, RecursiveCharacterTextSplitter, TokenTextSplitter | `MarkdownHeaderTextSplitter`, `HTMLHeaderTextSplitter`, code-aware splitters |
| **Vector store** | Chroma (persistent on disk) | Swap in Pinecone, Weaviate, Qdrant, FAISS via the same `VectorStore` interface |
| **Retrievers** | similarity, MMR, threshold | `MultiQueryRetriever`, `ContextualCompressionRetriever`, `EnsembleRetriever` for hybrid search |
| **Composition** | LCEL with `\|`, `RunnableParallel`, `RunnablePassthrough` | Add reranking, query rewriting, citation extraction — all as Runnables |
| **Memory** | `create_history_aware_retriever` + `create_retrieval_chain` | LangGraph for richer multi-turn / stateful flows |

**LCEL benefits**
- One interface (`.invoke / .stream / .batch / .ainvoke`) across every step
- Composition is just `|` — easy to read and easy to test piece by piece
- Streaming and batching come for free
- Each runnable can be inspected, mocked, or swapped without changing the rest

The notebook ends with a working `rag_chain` (stateless) and a `conversational_rag` (multi-turn). Either can be wired into a Gradio or Streamlit UI in a few lines.